In [ ]:
%pip install ipython-sql sqlalchemy pandas 

# Necessary Library And Data bases

In [2]:
%reload_ext sql
%sql sqlite://
%config SqlMagic.style = '_DEPRECATED_DEFAULT'


In [1]:

import pandas as pd
import sqlite3


In [8]:
patrons = pd.read_csv('data/patrons.csv')
books = pd.read_csv('data/books.csv')
checkouts = pd.read_csv('data/checkouts.csv')
customers= pd.read_csv('data/customers.csv')
orders = pd.read_csv('data/orders.csv') 



In [9]:
%sql --persist customers

 * sqlite://


'Persisted customers'

In [10]:
%sql --persist orders

 * sqlite://


'Persisted orders'

In [4]:
%sql --persist patrons

 * sqlite://


'Persisted patrons'

In [5]:
%sql --persist  books

 * sqlite://


'Persisted books'

In [6]:
%sql --persist  checkouts

 * sqlite://


'Persisted checkouts'

In [88]:

# Now connect your ipython-sql to this database
from sqlalchemy import create_engine

engine = create_engine('sqlite://')
patrons.to_sql('patrons', engine, index=False, if_exists='replace')
books.to_sql('books', engine, index=False, if_exists='replace')
checkouts.to_sql('checkouts', engine, index=False, if_exists='replace')

%sql sqlite://

# Basic



**Goals:**
- Understand databases and table structure
- Learn SQL basics to extract, organize, and analyze data
- Practice naming conventions and data types

We'll use a sample "library" database with patrons, books, and checkouts.


## What is SQL?

**SQL** (Structured Query Language) is a standard language used to manage and interact with relational databases.  
You can use SQL to:
- Extract data
- Organize information
- Analyze relationships between tables

A database contains tables (like spreadsheets):
- **Rows**: individual records (e.g., a library member)
- **Columns**: specific attributes (e.g., name, year joined)

## Naming Conventions

- **Table names:** lowercase, descriptive, use underscores  
  *Examples: patrons, books, checkouts*
- **Field (column) names:** lowercase, underscores, singular  
  *Examples: name, year_joined, total_fine*
- Each table should have a unique **primary key** (e.g., id)

**Data types:**
- `INTEGER` for whole numbers (years, ids)
- `TEXT` for strings (names, titles)
- `REAL` for decimal numbers (fines)

## Sample Data Structure

We'll use three tables:
- `patrons`: Library members
- `books`: Books in the library
- `checkouts`: Which patron checked out which book

Let's get started by loading the data!



1. SELECT – Retrieve Data
Explanation:
SELECT is used to fetch data from a table. You specify the columns you want.

Exercise:
Get all customer names and their ages.

In [11]:
%%sql
SELECT first_name, last_name, age FROM customers;


 * sqlite://
Done.


first_name,last_name,age
Alice,Smith,30
Bob,Johnson,25
Carol,Lee,35
David,Kim,28
Eve,Clark,40


2. WHERE – Filtering Rows
Explanation:
WHERE lets you filter rows based on a condition.

Exercise:
Find all customers who live in "Chicago".

In [13]:
%%sql
SELECT * FROM customers
WHERE city = 'Chicago';


 * sqlite://
Done.


index,customer_id,first_name,last_name,city,age
2,3,Carol,Lee,Chicago,35


3. ORDER BY – Sorting Results
Explanation:
ORDER BY sorts the result. Use ASC for ascending, DESC for descending.

Exercise:
List all orders sorted by amount, highest first.

In [14]:
%%sql
SELECT * FROM orders
ORDER BY amount DESC;

 * sqlite://
Done.


index,order_id,customer_id,order_date,amount
6,1007,5,2024-07-07,400
5,1006,2,2024-07-06,350
2,1003,1,2024-07-03,300
0,1001,1,2024-07-01,250
3,1004,3,2024-07-04,200
4,1005,4,2024-07-05,175
1,1002,2,2024-07-02,150


4. LIMIT – Restrict Number of Results
Explanation:
LIMIT returns only a certain number of rows.

In [16]:
%%sql
SELECT * FROM orders
LIMIT 3;


 * sqlite://
Done.


index,order_id,customer_id,order_date,amount
0,1001,1,2024-07-01,250
1,1002,2,2024-07-02,150
2,1003,1,2024-07-03,300


5. DISTINCT – Unique Values
Explanation:
DISTINCT removes duplicate rows from your result.

In [17]:
%%sql
SELECT DISTINCT city FROM customers;


 * sqlite://
Done.


city
New York
Los Angeles
Chicago
Houston
Phoenix


6. COUNT(), AVG(), SUM() – Simple Functions
Explanation:
These functions are used to aggregate data.

COUNT(*): Number of rows

AVG(column): Average value

SUM(column): Total sum

Exercise:
How many orders are there?

In [18]:
%%sql
SELECT COUNT(*) FROM orders;


 * sqlite://
Done.


COUNT(*)
7


In [19]:
%%sql
SELECT AVG(amount) FROM orders;


 * sqlite://
Done.


AVG(amount)
260.7142857142857


In [20]:
%%sql
SELECT SUM(amount) FROM orders;


 * sqlite://
Done.


SUM(amount)
1825


# Practice basic SQL Queries

In [7]:
%%sql
SELECT * FROM patrons;

 * sqlite://
Done.


index,id,name,year_joined,total_fine
0,1,James,2020,2.5
1,2,Izzy,2019,0.0
2,3,Maham,2021,1.25
3,4,Alice,2018,0.5


In [21]:
%%sql
SELECT customer_id, SUM(amount) AS total_amount
FROM orders
GROUP BY customer_id;


 * sqlite://
Done.


customer_id,total_amount
1,550
2,500
3,200
4,175
5,400


8. HAVING – Filter Groups
Explanation:
HAVING filters groups created by GROUP BY (useful for things you can’t do in WHERE).

Exercise:
Show customers who spent more than 400 in total.

7. GROUP BY – Summarize Data by Groups
Explanation:
GROUP BY groups rows with the same value in a column, so you can aggregate for each group.

Exercise:
Show total order amount per customer.

In [22]:
%%sql
SELECT customer_id, SUM(amount) AS total_spent
FROM orders
GROUP BY customer_id
HAVING total_spent > 400;


 * sqlite://
Done.


customer_id,total_spent
1,550
2,500


Exercise 1: Total order amount for each customer

In [23]:
%%sql
SELECT customer_id, SUM(amount) AS total_amount
FROM orders
GROUP BY customer_id;


 * sqlite://
Done.


customer_id,total_amount
1,550
2,500
3,200
4,175
5,400


Exercise 2: Show customers who have placed more than 1 order



In [24]:
%%sql
SELECT customer_id, COUNT(*) AS num_orders
FROM orders
GROUP BY customer_id
HAVING num_orders > 1;


 * sqlite://
Done.


customer_id,num_orders
1,2
2,2


Exercise 3: For each customer, list their name and total amount spent

In [25]:
%%sql
SELECT c.first_name, c.last_name, SUM(o.amount) AS total_spent
FROM customers c
INNER JOIN orders o ON c.customer_id = o.customer_id
GROUP BY c.customer_id;


 * sqlite://
Done.


first_name,last_name,total_spent
Alice,Smith,550
Bob,Johnson,500
Carol,Lee,200
David,Kim,175
Eve,Clark,400


Exercise 4: Show all orders along with the customer’s city

In [26]:
%%sql
SELECT o.order_id, o.amount, c.city
FROM orders o
INNER JOIN customers c ON o.customer_id = c.customer_id;


 * sqlite://
Done.


order_id,amount,city
1001,250,New York
1002,150,Los Angeles
1003,300,New York
1004,200,Chicago
1005,175,Houston
1006,350,Los Angeles
1007,400,Phoenix


Exercise 5: List all orders made by customers from ‘Los Angeles’

In [27]:
%%sql
SELECT o.*
FROM orders o
INNER JOIN customers c ON o.customer_id = c.customer_id
WHERE c.city = 'Los Angeles';


 * sqlite://
Done.


index,order_id,customer_id,order_date,amount
1,1002,2,2024-07-02,150
5,1006,2,2024-07-06,350


# Intermideate sql

In [3]:

products = pd.read_csv('data/products.csv')
customers2= pd.read_csv('data/customers2.csv')
orders2 = pd.read_csv('data/orders2.csv') 
orders_Item = pd.read_csv('data/order_items.csv') 




In [4]:
%sql --persist customers2

 * sqlite://


'Persisted customers2'

In [5]:
%sql --persist orders2

 * sqlite://


'Persisted orders2'

In [6]:
%sql --persist orders_Item

 * sqlite://


'Persisted orders_item'

In [7]:
%sql --persist products

 * sqlite://


'Persisted products'

## Advanced Filtering and Comparison Operators

A. IN, NOT IN
IN: Match a value in a list.

NOT IN: Exclude values in a list.

Example:
Find all customers from Seattle, Portland, or San Diego.

In [9]:
%%sql
Select * from customers2
where city in ('Seattle', 'Portland', 'San Diego')
limit 10;


 * sqlite://
Done.


index,customer_id,first_name,last_name,email,city,state,signup_date,is_active
0,1,Anna,Smith,anna.smith@email.com,Seattle,WA,2022-01-10,1
1,2,Brian,Lee,brian.lee@email.com,Portland,OR,2022-03-14,1
2,3,Carla,Kim,None,San Diego,CA,2021-09-18,0
3,4,David,Patel,david.patel@email.com,Seattle,WA,2023-02-25,1
5,6,Frank,Zhang,frank.zhang@email.com,Portland,OR,2022-08-21,1
7,8,Henry,Miller,henry.miller@email.com,San Diego,CA,2023-01-08,1
9,10,Jake,Singh,jake.singh@email.com,Seattle,WA,2023-04-17,1
10,11,Kim,Johnson,None,San Diego,CA,2022-06-13,1
11,12,Liam,Chen,liam.chen@email.com,Portland,OR,2022-12-19,1
12,13,Mia,Garcia,mia.garcia@email.com,Seattle,WA,2023-07-03,1


In [10]:
%%sql
Select * from customers2
where city Not in ('Seattle', 'Portland', 'San Diego')
limit 10;

 * sqlite://
Done.


index,customer_id,first_name,last_name,email,city,state,signup_date,is_active
4,5,Emma,Brown,emma.brown@email.com,Los Angeles,CA,2023-06-07,1
6,7,Grace,Gomez,grace.gomez@email.com,San Jose,CA,2021-11-11,1
8,9,Irene,O'Neil,irene.oneil@email.com,Los Angeles,CA,2022-05-15,0
13,14,Nina,Keller,nina.keller@email.com,Los Angeles,CA,2023-03-22,0
14,15,Omar,Ali,omar.ali@email.com,San Jose,CA,2022-09-14,1
18,19,Sara,Evans,None,Los Angeles,CA,2021-10-02,0


B. AND, OR :
Combine multiple conditions.

Example:
Find all active customers from California (CA) or Washington (WA).

In [17]:
%%sql
select * from customers2
WHERE (state = 'CA' OR state = 'WA') AND is_active = 1
limit 5;

 * sqlite://
Done.


index,customer_id,first_name,last_name,email,city,state,signup_date,is_active
0,1,Anna,Smith,anna.smith@email.com,Seattle,WA,2022-01-10,1
3,4,David,Patel,david.patel@email.com,Seattle,WA,2023-02-25,1
4,5,Emma,Brown,emma.brown@email.com,Los Angeles,CA,2023-06-07,1
6,7,Grace,Gomez,grace.gomez@email.com,San Jose,CA,2021-11-11,1
7,8,Henry,Miller,henry.miller@email.com,San Diego,CA,2023-01-08,1


C. BETWEEN:
Range for numbers or dates (inclusive).

Example:
Find orders placed between July 09 and July 14, 2023.

In [21]:
%%sql
select * from orders2
where  order_date BETWEEN '2023-07-09' AND '2023-07-14';

 * sqlite://
Done.


index,order_id,customer_id,order_date,order_status,shipped_date,shipping_city,shipping_state,total_amount
15,1016,16,2023-07-09,Shipped,2023-07-11,Seattle,WA,220.0
16,1017,17,2023-07-10,Shipped,2023-07-12,Portland,OR,18.0
17,1018,18,2023-07-12,Shipped,2023-07-13,San Diego,CA,115.0
18,1019,19,2023-07-13,Pending,None,Los Angeles,CA,9.0
19,1020,20,2023-07-14,Shipped,2023-07-15,Seattle,WA,85.0


D. LIKE, Pattern Matching
% = wildcard for any sequence of characters.

_ = wildcard for a single character.

Example:
Find all products with "Desk" in the name

In [30]:
%%sql
SELECT product_id, product_name
FROM products
WHERE product_name LIKE '%Desk%';


 * sqlite://
Done.


product_id,product_name
5,Desk Lamp
7,Standing Desk


E. **CASE WHEN** in SQL is used for conditional logic, similar to "if-else" in programming.
It lets you create new columns or values based on conditions.

Example:
Show a customer activity label ("Active" or "Inactive") for each customer.

In [39]:
%%sql
select first_name, last_name, 
case when is_active = 1 then 'Active' else 'Inactive' end as activity_status
from customers2
limit 5;


 * sqlite://
Done.


first_name,last_name,activity_status
Anna,Smith,Active
Brian,Lee,Active
Carla,Kim,Inactive
David,Patel,Active
Emma,Brown,Active


F. **Comparison Operators (=, <>, >, <, >=, <=)**

Example:
Find all products priced above $30 and not in "Electronics".

In [40]:
%%sql
select * from products
where price > 30 AND category <> 'Electronics';

 * sqlite://
Done.


index,product_id,product_name,category,price,stock
4,5,Desk Lamp,Home,35.0,60
5,6,Running Shoes,Fitness,75.0,30
6,7,Standing Desk,Home,250.0,15
11,12,Blender,Kitchen,60.0,35
13,14,Air Purifier,Home,150.0,20
17,18,Floor Lamp,Home,80.0,24
18,19,Espresso Machine,Kitchen,300.0,8


##  Practice Exercises (Filtering & Comparison) 

1. List all customers from Portland or San Diego who are active.

2. Show all orders with a total amount between $30 and $100.

3. Get all products in the "Fitness" or "Home" categories with price <= $35.

4. Find all orders not shipped to California ("CA").

5. List all customers whose last name starts with "B".

6. Show all products that do not have the word "Bluetooth" in their name.

7. Display all orders placed before August 7, 2023.

8. List all products with "Coffee" anywhere in the name.

9. Show customers who signed up between January 1, 2022 and June 30, 2023.

10. For each customer, show their name and a label "Recent" if they signed up after 2023-01-01, otherwise "Long-term". 

## Sample Solution




``` -- 1. 
SELECT first_name, last_name, city
FROM customers
WHERE city IN ('Portland', 'San Diego') AND is_active = 1; 

-- 2.
SELECT order_id, total_amount
FROM orders
WHERE total_amount BETWEEN 30 AND 100;

-- 3.
SELECT product_name, category, price
FROM products
WHERE category IN ('Fitness', 'Home') AND price <= 35;

-- 4.
SELECT order_id, shipping_state
FROM orders
WHERE shipping_state <> 'CA';

-- 5.
SELECT first_name, last_name
FROM customers
WHERE last_name LIKE 'B%';

-- 6.
SELECT product_id, product_name
FROM products
WHERE product_name NOT LIKE '%Bluetooth%';

-- 7.
SELECT order_id, order_date
FROM orders
WHERE order_date < '2023-08-07';

-- 8.
SELECT product_id, product_name
FROM products
WHERE product_name LIKE '%Coffee%';

-- 9.
SELECT first_name, last_name, signup_date
FROM customers
WHERE signup_date BETWEEN '2022-01-01' AND '2023-06-30';

-- 10.
SELECT first_name, last_name,
  CASE WHEN signup_date > '2023-01-01' THEN 'Recent'
       ELSE 'Long-term'
  END AS signup_status
FROM customers; 

## 2. Handling NULL Values in SQL